# HerWILL × UAP Datathon 2026 — Toxicity Classification: K-Fold Stacked Ensemble
**One self-contained, resumable notebook.** Needs only the raw `train.csv` / `test.csv`. Uses **no external data** —
only publicly released pretrained checkpoints (explicitly allowed by the competition rules) fine-tuned on the provided training set.

## System architecture (this is an ensemble *system*, not a single model)

```
                       ┌─► TF-IDF (word 1-2g + char 3-5g) ─► LinearSVC + Platt calibration ───────┐
                       │        5-fold out-of-fold (OOF) probabilities                              │
 raw text ─► clean ────┼─► XLM-RoBERTa-large  (≈560M params)  4-fold OOF, LLRD, label smoothing ───┤
 (Unicode NFC,         │                                                                            ├─► STACKER ─► per-class
  URL/@ masking)       ├─► MuRIL-base         (≈237M params)  5-fold OOF, LLRD + EMA, noise-aware ──┤   (LogReg / HistGB on   thresholds ─► label
                       │                                                                            │    log-probs + text
                       └─► mDeBERTa-v3-base   (≈280M params)  5-fold OOF, LLRD + EMA, noise-aware ──┘    meta-features)
```

## Why it is built this way
| Design choice | Reason |
|---|---|
| **K-fold out-of-fold predictions** instead of one train/val split | Every labeled row (≈47k) gets an honest prediction from a model that never saw it; blend/stacker/thresholds are tuned on ≈47k rows instead of 7k, and all models train on 67–80 % of the data instead of 85 % with the rest wasted. |
| **Different backbones** (XLM-R, MuRIL, mDeBERTa) + lexical SVM | Three seeds of one model differed by only 0.0014 F1 — they make the same mistakes. Different pretraining corpora/tokenizers make *different* mistakes, which is what ensembling needs. |
| **Layer-wise LR decay (LLRD)**, warmup, linear decay | Lower layers hold general language knowledge and should move less; stabilises large-model fine-tuning. |
| **EMA of weights** (base-size models) | Averages the noisy late-training trajectory; usually a free +0.3–0.8 F1. Disabled for XLM-R-large (T4 memory). |
| **Label smoothing + class-balanced loss** | The labels are noisy (e.g. "I enjoyed this thoroughly" is labeled *Subtly Toxic*) and the Neutral class is the 12 % minority. |
| **Noise-aware sample weights** (later stages) | Training rows that the *out-of-fold* XLM-R confidently contradicts are down-weighted (×0.3), never dropped and never relabeled by hand. |
| **Stacker + per-class thresholds** | Learns how much to trust each model per class; thresholds directly maximise Macro-F1 (argmax does not). Thresholds are evaluated with *nested* CV so the reported score is not inflated. |

## Rules compliance
* Only the provided Kaggle dataset is used for training; pretrained checkpoints only (allowed: *"Use of pre-trained models is allowed, provided no external training data is used"*).
* No hand labeling / human prediction of validation or test rows anywhere. `test.csv` is used only to produce predictions (and, for the SVM/TF-IDF, is never used to fit vocabulary).
* Fully reproducible: fixed seeds, fold assignment saved to disk, every hyper-parameter printed in the log and in the final methodology table.

## How to run
1. Set `SMOKE_TEST = True` in the config cell and **Run all** (~10–15 min, downloads the 3 checkpoints once). This validates every stage end-to-end on a tiny subset.
2. Set `SMOKE_TEST = False`, set `HOURS_LEFT` to your real remaining time, **Run all**.
3. If Colab disconnects: reconnect, update `HOURS_LEFT`, **Run all** again — finished folds are loaded from Google Drive and skipped.

## 0. Environment

In [1]:
import subprocess, sys
# sentencepiece/protobuf are needed to build the XLM-R / mDeBERTa tokenizers on some installs
for _pkg in ["sentencepiece", "protobuf"]:
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _pkg], check=False,
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    except Exception as _e:
        print("pip install skipped:", _pkg, _e)
print("environment ready")

environment ready


## 1. Configuration — every hyper-parameter of the system lives here
Nothing is hidden in the code below: this cell + the logs are the complete specification.

In [2]:
import os, sys, re, gc, json, math, time, random, shutil, inspect, unicodedata, warnings, traceback
from pathlib import Path
from datetime import datetime

# ─────────────────────────── run control ───────────────────────────
SMOKE_TEST     = False   # True → tiny end-to-end dry run of ALL stages (do this first!)
HOURS_LEFT     = 9.0     # wall-clock hours until the Kaggle deadline (update after a restart)
SAFETY_MINUTES = 40      # reserved at the end for stacking + writing submissions
SEED           = 42
RUN = {"svm": True, "xlmr_large": True, "muril": True, "mdeberta": True}   # switch stages on/off
USE_NOISE_DOWNWEIGHT = True     # label-noise-aware sample weights in later stages (see section 6)
NOISE_WEIGHT         = 0.30     # weight given to rows the OOF XLM-R confidently contradicts
NOISE_MAX_FRACTION   = 0.06     # never down-weight more than 6 % of the rows
FORCE_RETUNE_THRESHOLDS = False # persistent-threshold protocol: reuse saved thresholds unless True

# ─────────────────────────── data / cleaning (same rules as Block 1) ───────────────────────────
MAX_CHAR_LEN = 2000
CLASS_IDS    = [0, 1, 2]
N_CLASSES    = 3
LABEL_NAMES  = {0: "Explicitly Toxic", 1: "Subtly Toxic", 2: "Non-toxic/Neutral"}
FOLD_SCHEMES = [2, 3, 4, 5]     # fold columns created once, saved to disk, reused by every model

# ─────────────────────────── classical model (Stage 1) ───────────────────────────
SVM_CFG = dict(n_folds=5, word_ngrams=(1, 2), char_ngrams=(3, 5), max_features_each=60_000,
               min_df=2, C_grid=[0.1, 0.3, 1.0], class_weight="balanced", calibration="sigmoid (Platt), cv=3")

# ─────────────────────────── transformer models (Stages 2-4) ───────────────────────────
# lr           : learning rate of the TOP encoder layer;  lower layers get lr * llrd**(depth from top)
# head_lr_mult : classifier head learning-rate multiplier      ema_decay: None = EMA off
# est_min_per_fold : rough T4 minutes per fold (used ONLY to decide whether a stage fits in the time left)
MODELS = {
    "xlmr_large": dict(hf_name="xlm-roberta-large", pretty="XLM-RoBERTa-large",
        n_folds=4, epochs=4, batch_size=16, grad_accum=2, eval_bs=64, max_len=128,
        lr=2e-5, llrd=0.95, head_lr_mult=2.0, weight_decay=0.01, warmup_frac=0.08,
        label_smoothing=0.05, ema_decay=None, fgm=False, fp16=True,
        noise_aware=False, probe=False, est_min_per_fold=38),
    "muril": dict(hf_name="google/muril-base-cased", pretty="MuRIL-base-cased",
        n_folds=5, epochs=3, batch_size=32, grad_accum=1, eval_bs=128, max_len=128,
        lr=3e-5, llrd=0.90, head_lr_mult=2.0, weight_decay=0.01, warmup_frac=0.08,
        label_smoothing=0.05, ema_decay=0.999, fgm=False, fp16=True,
        noise_aware=True, probe=False, est_min_per_fold=13),
    "mdeberta": dict(hf_name="microsoft/mdeberta-v3-base", pretty="mDeBERTa-v3-base",
        n_folds=5, epochs=3, batch_size=32, grad_accum=1, eval_bs=128, max_len=128,
        lr=3e-5, llrd=0.90, head_lr_mult=2.0, weight_decay=0.01, warmup_frac=0.08,
        label_smoothing=0.05, ema_decay=0.999, fgm=False, fp16=True,
        noise_aware=True, probe=True, est_min_per_fold=18),
}
# fgm=True adds fast-gradient adversarial training on the word embeddings (≈ +70 % time/fold) — off by default.

COLLAPSE_F1   = 0.42    # if epoch-1 sub-holdout Macro-F1 is below this the fold is restarted (rare XLM-R-large collapse)
EPOCH_EVAL_N  = 3000    # rows of the holdout fold used for the per-epoch diagnostic

if SMOKE_TEST:          # tiny dry run: 2 folds, 1 epoch, ~1000 rows, short sequences
    HOURS_LEFT = 50.0
    for _k, _c in MODELS.items():
        _c.update(n_folds=2, epochs=1, max_len=64, est_min_per_fold=1)
    SVM_CFG.update(n_folds=2, max_features_each=5000, C_grid=[0.3])
    COLLAPSE_F1 = -1.0
    RUN.update({k: True for k in RUN})

## 2. Utilities — logging, seeding, work directory (Google Drive), timing, parameter counting

In [3]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 110); pd.set_option("display.width", 200)

T0 = time.time()
DEADLINE = T0 + HOURS_LEFT * 3600 - SAFETY_MINUTES * 60
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

# ── work directory: Google Drive if available (survives runtime disconnects) ──
def setup_workdir():
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        base = Path("/content/drive/MyDrive/datathon_run")
    except Exception as e:
        print("Google Drive not mounted (", type(e).__name__, ") → using local /content/datathon_run")
        base = Path("/content/datathon_run") if Path("/content").exists() else Path("./datathon_run")
    if SMOKE_TEST: base = base / "smoke"
    base.mkdir(parents=True, exist_ok=True)
    (base / "folds").mkdir(exist_ok=True); (base / "submissions").mkdir(exist_ok=True)
    return base

WORK = setup_workdir()
LOGFILE = WORK / "run_log.txt"

def log(msg="", level="INFO"):
    el = time.time() - T0
    line = f"[{datetime.now().strftime('%H:%M:%S')} | +{int(el//3600)}h{int(el%3600//60):02d}m] {level:<5} {msg}"
    print(line, flush=True)
    try:
        with open(LOGFILE, "a", encoding="utf-8") as f: f.write(line + "\n")
    except Exception: pass

def banner(title):
    log("=" * 88); log(title); log("=" * 88)

def time_left_s(): return DEADLINE - time.time()

def gpu_mem():
    if DEVICE != "cuda": return "cpu"
    return f"{torch.cuda.memory_allocated()/2**30:.1f}/{torch.cuda.get_device_properties(0).total_memory/2**30:.1f} GB"

def cleanup():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

def count_params(model):
    tot = sum(p.numel() for p in model.parameters())
    trn = sum(p.numel() for p in model.parameters() if p.requires_grad)
    emb = sum(p.numel() for n, p in model.named_parameters() if "word_embeddings" in n)
    head = sum(p.numel() for n, p in model.named_parameters()
               if n.startswith(("classifier", "score", "pooler")) or ".pooler." in n)
    return dict(total=tot, trainable=trn, embeddings=emb, head=head, encoder=tot - emb - head)

def log_param_table(key, model, cfg):
    pc = count_params(model)
    log(f"PARAMS  {cfg['pretty']:<20} total={pc['total']/1e6:8.1f}M | trainable={pc['trainable']/1e6:8.1f}M | "
        f"token-embeddings={pc['embeddings']/1e6:7.1f}M | transformer-layers={pc['encoder']/1e6:7.1f}M | head={pc['head']/1e6:6.2f}M")
    return pc

log(f"torch {torch.__version__} | device={DEVICE}" +
    (f" | GPU={torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory/2**30:.1f} GB)" if DEVICE=="cuda" else ""))
log(f"work dir: {WORK} | smoke_test={SMOKE_TEST} | hours_left={HOURS_LEFT} | deadline for training stages in {time_left_s()/3600:.2f}h")
if DEVICE == "cpu" and not SMOKE_TEST:
    log("No GPU detected! Runtime → Change runtime type → T4 GPU. Training will be extremely slow.", "WARN")

Google Drive not mounted ( ValueError ) → using local /content/datathon_run
[04:18:29 | +0h02m] INFO  torch 2.11.0+cu128 | device=cuda | GPU=Tesla T4 (14.6 GB)
[04:18:29 | +0h02m] INFO  work dir: /content/datathon_run | smoke_test=False | hours_left=9.0 | deadline for training stages in 8.30h


## 3. Data — load raw CSVs, clean, filter, create fold assignments
Cleaning philosophy (same rules as Block 1): remove noise, keep signal. URLs/@mentions → tokens, Unicode NFC (important for Bangla
conjuncts), control chars/whitespace normalised — but **emoji, casing, punctuation repetition and script mixing are kept** because they carry toxicity signal.
Rows with conflicting duplicate labels, empty text or absurd length are dropped from *training only*; every test row is kept.

In [4]:
def find_file(name):
    for d in [Path("/content"), Path("/content/data"), Path("/content/drive/MyDrive"), WORK, Path("."), Path("/mnt/user-data/uploads")]:
        if (d / name).exists(): return d / name
    raise FileNotFoundError(f"{name} not found — upload train.csv/test.csv to /content (or Drive) and re-run.")

URL_RE, MENTION_RE = re.compile(r"https?://\S+|www\.\S+"), re.compile(r"@\w+")
WS_RE, CTRL_RE = re.compile(r"\s+"), re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]")

def clean_text(raw):
    if not isinstance(raw, str): return ""
    t = unicodedata.normalize("NFC", raw)
    t = URL_RE.sub(" [URL] ", t); t = MENTION_RE.sub(" [USER] ", t)
    t = CTRL_RE.sub(" ", t)
    return WS_RE.sub(" ", t).strip()

def script_type(t):
    b = len(re.findall(r"[\u0980-\u09FF]", t)); l = len(re.findall(r"[a-zA-Z]", t))
    return "mixed" if (b and l) else "bangla" if b else "latin" if l else "other"

def prepare_data():
    cache_tr, cache_te = WORK / "train_clean.csv", WORK / "test_clean.csv"
    if cache_tr.exists() and cache_te.exists():
        log("loading cached cleaned data + fold assignment from work dir (guarantees identical folds after a restart)")
        return pd.read_csv(cache_tr, keep_default_na=False, na_values=[]), pd.read_csv(cache_te, keep_default_na=False, na_values=[])
    tr_raw, te_raw = pd.read_csv(find_file("train.csv")), pd.read_csv(find_file("test.csv"))
    log(f"raw train {tr_raw.shape} | raw test {te_raw.shape} | columns {list(tr_raw.columns)}")
    vc = tr_raw["y"].value_counts(normalize=True).sort_index()
    log("raw class balance: " + " | ".join(f"{LABEL_NAMES[k]}={v:.3f}" for k, v in vc.items()))
    tr = tr_raw.copy(); tr["text"] = tr["text"].astype(str)
    te = te_raw.copy(); te["text"] = te["text"].astype(str)
    grp = tr[tr["text"].duplicated(keep=False)].groupby("text")["y"].nunique()
    conflict = set(grp[grp > 1].index)
    tr["text_c"] = tr["text"].apply(clean_text); te["text_c"] = te["text"].apply(clean_text)
    m_empty = tr["text_c"].str.len() == 0
    m_long = tr["text_c"].str.len() > MAX_CHAR_LEN
    m_conf = tr["text"].isin(conflict) & tr["text"].duplicated(keep="first")
    log(f"dropping from TRAIN only: empty={int(m_empty.sum())} | >{MAX_CHAR_LEN} chars={int(m_long.sum())} | conflicting-label duplicates={int(m_conf.sum())}")
    tr = tr[~(m_empty | m_long | m_conf)].reset_index(drop=True)
    def finish(d):
        o = pd.DataFrame({"id": d["id"], "text": d["text_c"]})
        o["char_len"] = o["text"].str.len(); o["word_count"] = o["text"].str.split().apply(len)
        o["script"] = o["text"].apply(script_type)
        return o
    tr_out, te_out = finish(tr), finish(te)
    tr_out["y"] = tr["y"].values
    if SMOKE_TEST:   # tiny stratified subset
        from sklearn.model_selection import train_test_split
        _, keep = train_test_split(np.arange(len(tr_out)), test_size=1000, stratify=tr_out["y"], random_state=SEED)
        tr_out = tr_out.iloc[np.sort(keep)].reset_index(drop=True); te_out = te_out.iloc[:200].reset_index(drop=True)
    for k in FOLD_SCHEMES:   # one fold column per scheme, created ONCE and saved
        col = np.zeros(len(tr_out), dtype=int)
        for f, (_, ho) in enumerate(StratifiedKFold(k, shuffle=True, random_state=SEED + k).split(tr_out, tr_out["y"])):
            col[ho] = f
        tr_out[f"fold{k}"] = col
    tr_out.to_csv(cache_tr, index=False); te_out.to_csv(cache_te, index=False)
    return tr_out, te_out

train_df, test_df = prepare_data()
for d in (train_df, test_df): d["text"] = d["text"].fillna("").astype(str)
y_all = train_df["y"].to_numpy().astype(int)
N_TRAIN, N_TEST = len(train_df), len(test_df)
log(f"cleaned train rows={N_TRAIN:,} | test rows={N_TEST:,}")
log("class balance after cleaning: " + " | ".join(f"{LABEL_NAMES[k]}={v:.3f}" for k, v in pd.Series(y_all).value_counts(normalize=True).sort_index().items()))
log("script mix (train): " + str(train_df["script"].value_counts().to_dict()))
PRIOR = np.bincount(y_all, minlength=3) / len(y_all)

[04:18:35 | +0h02m] INFO  raw train (47817, 3) | raw test (11955, 2) | columns ['id', 'text', 'y']
[04:18:35 | +0h02m] INFO  raw class balance: Explicitly Toxic=0.497 | Subtly Toxic=0.380 | Non-toxic/Neutral=0.123
[04:18:36 | +0h02m] INFO  dropping from TRAIN only: empty=0 | >2000 chars=2 | conflicting-label duplicates=78
[04:18:38 | +0h02m] INFO  cleaned train rows=47,737 | test rows=11,955
[04:18:38 | +0h02m] INFO  class balance after cleaning: Explicitly Toxic=0.497 | Subtly Toxic=0.380 | Non-toxic/Neutral=0.124
[04:18:38 | +0h02m] INFO  script mix (train): {'latin': 27565, 'bangla': 19136, 'mixed': 994, 'other': 42}


## 4. Metrics & threshold tooling (shared by every stage; includes the persistent-threshold protocol)

In [5]:
def fast_macro_f1(y, pred, k=3):
    cm = np.bincount(np.asarray(y) * k + np.asarray(pred), minlength=k * k).reshape(k, k)
    tp = np.diag(cm).astype(float); fp = cm.sum(0) - tp; fn = cm.sum(1) - tp
    den = 2 * tp + fp + fn
    return float(np.where(den > 0, 2 * tp / np.maximum(den, 1), 0).mean())

def softmax_np(z):
    z = z - z.max(1, keepdims=True); e = np.exp(z); return e / e.sum(1, keepdims=True)

def report(y, probs, title):
    pred = probs.argmax(1); f1 = fast_macro_f1(y, pred)
    log(f"{title}: Macro-F1 = {f1:.4f} | accuracy = {(pred == y).mean():.4f}")
    per = f1_score(y, pred, average=None, labels=CLASS_IDS)
    log("   per-class F1: " + " | ".join(f"{LABEL_NAMES[c]}={per[c]:.4f}" for c in CLASS_IDS))
    cm = confusion_matrix(y, pred, labels=CLASS_IDS)
    log("   confusion (rows=true, cols=pred): " + " || ".join(f"{LABEL_NAMES[c][:8]}:{cm[c].tolist()}" for c in CLASS_IDS))
    return f1

def tune_class_weights(probs, y, seed=0):
    '''Coarse-to-fine grid search over 2 log-multipliers (class-0 multiplier fixed at 1) maximising Macro-F1.
       Macro-F1 is piecewise constant, so a grid is more reliable than gradient-free simplex methods.'''
    lp = np.log(np.clip(probs, 1e-9, 1))
    best, bw = -1, (0.0, 0.0)
    for a in np.arange(-1.0, 1.0001, 0.05):
        for b in np.arange(-1.0, 1.0001, 0.05):
            f = fast_macro_f1(y, (lp + np.array([0, a, b])).argmax(1))
            if f > best: best, bw = f, (a, b)
    for a in np.arange(bw[0] - 0.05, bw[0] + 0.0501, 0.01):
        for b in np.arange(bw[1] - 0.05, bw[1] + 0.0501, 0.01):
            f = fast_macro_f1(y, (lp + np.array([0, a, b])).argmax(1))
            if f > best: best, bw = f, (a, b)
    return np.exp(np.array([0, bw[0], bw[1]])), best

def apply_weights(probs, w): return (np.log(np.clip(probs, 1e-9, 1)) + np.log(w)).argmax(1)

def nested_threshold_estimate(probs, y, n_splits=5, seed=123):
    '''Honest estimate: tune thresholds on 4/5 of the OOF rows, score on the remaining 1/5, concatenate.'''
    pred = np.zeros(len(y), dtype=int)
    for tr, te in StratifiedKFold(n_splits, shuffle=True, random_state=seed).split(probs, y):
        w, _ = tune_class_weights(probs[tr], y[tr]); pred[te] = apply_weights(probs[te], w)
    return fast_macro_f1(y, pred)

THR_PATH = WORK / "thresholds.json"
def persistent_thresholds(probs, y, key):
    '''Persistent-threshold protocol: tune ONCE per configuration key, save, and reuse on every later run
       (unless FORCE_RETUNE_THRESHOLDS) so comparisons/reruns are not silently re-tuned in their own favour.'''
    store = json.loads(THR_PATH.read_text()) if THR_PATH.exists() else {}
    if key in store and not FORCE_RETUNE_THRESHOLDS:
        w = np.array(store[key]); log(f"using PERSISTED thresholds '{key}': {np.round(w, 3).tolist()}")
    else:
        w, f = tune_class_weights(probs, y); store[key] = w.tolist(); THR_PATH.write_text(json.dumps(store, indent=1))
        log(f"tuned + PERSISTED thresholds '{key}': {np.round(w, 3).tolist()} (in-sample F1 {f:.4f})")
    return w

## 5. Stage 1 — TF-IDF + calibrated LinearSVC (5-fold OOF)
Cheap, lexical, and *wrong in different places* than the transformers — that diversity is why it still earns blend weight despite a low solo score.
TF-IDF is fitted **inside each fold on that fold's training rows only** (no vocabulary/IDF information from the held-out rows or from `test.csv`).

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from scipy.sparse import hstack

def fold_files(key, f):
    return WORK / "folds" / f"{key}_f{f}_oof.npy", WORK / "folds" / f"{key}_f{f}_idx.npy", \
           WORK / "folds" / f"{key}_f{f}_test.npy", WORK / "folds" / f"{key}_f{f}_meta.json"

def fold_done(key, f): return all(p.exists() for p in fold_files(key, f))

def run_svm_stage():
    banner("STAGE 1 — TF-IDF + calibrated LinearSVC")
    if not RUN["svm"]: log("stage disabled → skipped"); return
    K, cfg = SVM_CFG["n_folds"], SVM_CFG
    log("SVM config: " + json.dumps({k: str(v) for k, v in cfg.items()}))
    try:
        fcol = f"fold{K}"; texts = train_df["text"]; test_texts = test_df["text"]
        def vec():
            return (TfidfVectorizer(analyzer="word", ngram_range=cfg["word_ngrams"], min_df=cfg["min_df"], max_features=cfg["max_features_each"], sublinear_tf=True),
                    TfidfVectorizer(analyzer="char_wb", ngram_range=cfg["char_ngrams"], min_df=cfg["min_df"], max_features=cfg["max_features_each"], sublinear_tf=True))
        best_C_path = WORK / "svm_best_C.json"
        if best_C_path.exists(): C = json.loads(best_C_path.read_text())["C"]; log(f"SVM C loaded from disk: {C}")
        else:
            tr = np.where(train_df[fcol] != 0)[0]; ho = np.where(train_df[fcol] == 0)[0]
            wv, cv_ = vec(); Xtr = hstack([wv.fit_transform(texts.iloc[tr]), cv_.fit_transform(texts.iloc[tr])]).tocsr()
            Xho = hstack([wv.transform(texts.iloc[ho]), cv_.transform(texts.iloc[ho])]).tocsr()
            scores = {}
            for c in cfg["C_grid"]:
                m = LinearSVC(C=c, class_weight=cfg["class_weight"], max_iter=5000, random_state=SEED).fit(Xtr, y_all[tr])
                scores[c] = fast_macro_f1(y_all[ho], m.decision_function(Xho).argmax(1)); log(f"   C={c}: fold-0 Macro-F1 (uncalibrated argmax) = {scores[c]:.4f}")
            C = max(scores, key=scores.get); best_C_path.write_text(json.dumps({"C": C, "scores": {str(k): v for k, v in scores.items()}}))
            log(f"selected C = {C}")
        for f in range(K):
            if fold_done("svm", f): log(f"fold {f+1}/{K}: already done → skipped"); continue
            t0 = time.time(); tr = np.where(train_df[fcol] != f)[0]; ho = np.where(train_df[fcol] == f)[0]
            wv, cv_ = vec()
            Xtr = hstack([wv.fit_transform(texts.iloc[tr]), cv_.fit_transform(texts.iloc[tr])]).tocsr()
            Xho = hstack([wv.transform(texts.iloc[ho]), cv_.transform(texts.iloc[ho])]).tocsr()
            Xte = hstack([wv.transform(test_texts), cv_.transform(test_texts)]).tocsr()
            base = LinearSVC(C=C, class_weight=cfg["class_weight"], max_iter=5000, random_state=SEED)
            clf = CalibratedClassifierCV(base, method="sigmoid", cv=3).fit(Xtr, y_all[tr])
            p_ho, p_te = clf.predict_proba(Xho), clf.predict_proba(Xte)
            nfeat = Xtr.shape[1]; nparam = nfeat * N_CLASSES + N_CLASSES
            f1 = fast_macro_f1(y_all[ho], p_ho.argmax(1))
            log(f"fold {f+1}/{K}: features={nfeat:,} | linear-model params≈{nparam:,} (x3 calibrated copies) | holdout Macro-F1={f1:.4f} | {(time.time()-t0)/60:.1f} min")
            po, pi, pt, pm = fold_files("svm", f)
            np.save(po, p_ho.astype(np.float32)); np.save(pi, ho); np.save(pt, p_te.astype(np.float32))
            pm.write_text(json.dumps(dict(f1=f1, params=nparam, minutes=(time.time()-t0)/60, C=C)))
        report_model("svm", K, "TF-IDF + SVM")
    except Exception:
        log("SVM stage FAILED:\n" + traceback.format_exc(), "ERROR")

def load_model_outputs(key, n_folds):
    '''Assemble OOF matrix (N,3) and fold-averaged test probs. Returns (oof, test_mean, complete?)'''
    oof = np.full((N_TRAIN, N_CLASSES), np.nan, dtype=np.float32); tests = []; done = 0
    for f in range(n_folds):
        if not fold_done(key, f): continue
        po, pi, pt, _ = fold_files(key, f)
        oof[np.load(pi)] = np.load(po); tests.append(np.load(pt)); done += 1
    return oof, (np.mean(tests, 0) if tests else None), done == n_folds

def report_model(key, n_folds, title):
    oof, te, complete = load_model_outputs(key, n_folds)
    if not complete: log(f"{title}: only {int(np.sum(~np.isnan(oof[:,0])))} rows covered → incomplete, excluded from the stack", "WARN"); return None
    f1 = report(y_all, oof, f"{title} — full OOF over {N_TRAIN:,} rows")
    return f1

run_svm_stage()

[04:18:50 | +0h02m] INFO  ========================================================================================
[04:18:50 | +0h02m] INFO  STAGE 1 — TF-IDF + calibrated LinearSVC
[04:18:50 | +0h02m] INFO  ========================================================================================
[04:18:50 | +0h02m] INFO  SVM config: {"n_folds": "5", "word_ngrams": "(1, 2)", "char_ngrams": "(3, 5)", "max_features_each": "60000", "min_df": "2", "C_grid": "[0.1, 0.3, 1.0]", "class_weight": "balanced", "calibration": "sigmoid (Platt), cv=3"}
[04:18:58 | +0h02m] INFO     C=0.1: fold-0 Macro-F1 (uncalibrated argmax) = 0.5298
[04:19:03 | +0h02m] INFO     C=0.3: fold-0 Macro-F1 (uncalibrated argmax) = 0.5281
[04:19:08 | +0h02m] INFO     C=1.0: fold-0 Macro-F1 (uncalibrated argmax) = 0.5087
[04:19:08 | +0h02m] INFO  selected C = 0.1
[04:19:19 | +0h02m] INFO  fold 1/5: features=108,399 | linear-model params≈325,200 (x3 calibrated copies) | holdout Macro-F1=0.5123 | 0.2 min
[04:19:31 | +0h03m] INF

## 6. Transformer engine
A compact, dependency-light PyTorch training loop (instead of `Trainer`, whose arguments changed across `transformers` versions and cost ~2 min per checkpoint write).
Features: dynamic padding + length-bucketed batching (≈1.5× faster on short toxic comments), fp16 autocast, gradient accumulation, LLRD, warmup + linear decay,
optional EMA / FGM, class-balanced + label-smoothed + per-sample-weighted loss, collapse/NaN/OOM guards with automatic retry, and resumable per-fold outputs.
No checkpoints are written to disk — a fold is trained for a fixed number of epochs and its predictions are saved.

In [7]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

class CollapseError(RuntimeError): pass

# ── model / tokenizer factories (separate functions so they can be swapped in tests) ──
def load_tokenizer(cfg):
    try: return AutoTokenizer.from_pretrained(cfg["hf_name"])
    except Exception as e:
        log(f"fast tokenizer load failed ({type(e).__name__}); retrying with use_fast=False", "WARN")
        return AutoTokenizer.from_pretrained(cfg["hf_name"], use_fast=False)

def load_model(cfg):
    m = AutoModelForSequenceClassification.from_pretrained(cfg["hf_name"], num_labels=N_CLASSES)
    return m.float()   # force fp32 master weights (fp16 autocast handles the math)

def tokenize_texts(tok, texts, max_len):
    out = []
    for s in range(0, len(texts), 4000):
        enc = tok(texts[s:s+4000], truncation=True, max_length=max_len, add_special_tokens=True,
                  return_attention_mask=False, return_token_type_ids=False)
        out.extend(enc["input_ids"])
    return out

# ── batching ──
def make_train_batches(lengths, idx, bs, rng, chunk_mult=64):
    idx = np.array(idx); rng.shuffle(idx); batches = []
    for s in range(0, len(idx), bs * chunk_mult):
        c = idx[s:s + bs * chunk_mult]; c = c[np.argsort(lengths[c], kind="stable")]
        batches += [c[b:b + bs] for b in range(0, len(c), bs)]
    order = rng.permutation(len(batches)); return [batches[i] for i in order]

def collate(ids_list, batch_idx, pad_id, device):
    seqs = [ids_list[i] for i in batch_idx]; L = max(len(s) for s in seqs)
    inp = np.full((len(seqs), L), pad_id, dtype=np.int64); att = np.zeros((len(seqs), L), dtype=np.int64)
    for r, s in enumerate(seqs): inp[r, :len(s)] = s; att[r, :len(s)] = 1
    return torch.from_numpy(inp).to(device), torch.from_numpy(att).to(device)

@torch.no_grad()
def predict_logits(model, ids_list, indices, pad_id, eval_bs, use_amp):
    model.eval(); indices = np.asarray(indices)
    lens = np.array([len(ids_list[i]) for i in indices]); order = np.argsort(lens, kind="stable")
    out = np.zeros((len(indices), N_CLASSES), dtype=np.float32)
    for s in range(0, len(order), eval_bs):
        pos = order[s:s + eval_bs]; inp, att = collate(ids_list, indices[pos], pad_id, DEVICE)
        with torch.autocast(device_type=DEVICE, dtype=torch.float16, enabled=use_amp):
            lg = model(input_ids=inp, attention_mask=att).logits
        out[pos] = lg.float().cpu().numpy()
    return out

# ── loss ──
def weighted_ce(logits, y, cw, sw, smoothing):
    '''class-balanced + label-smoothed CE with per-sample weights, normalised like torch's weighted mean.'''
    per = F.cross_entropy(logits, y, weight=cw, label_smoothing=smoothing, reduction="none")
    return (per * sw).sum() / (cw[y] * sw).sum().clamp_min(1e-8)

# ── optimizer with layer-wise LR decay ──
def build_optimizer(model, cfg, lr_scale=1.0):
    n_layers = model.config.num_hidden_layers; top = cfg["lr"] * lr_scale; groups = {}
    for n, p in model.named_parameters():
        if not p.requires_grad: continue
        m = re.search(r"\.layer\.(\d+)\.", n)
        if n.startswith(("classifier", "score", "pooler")) or ".pooler." in n:
            lr, tag = top * cfg["head_lr_mult"], "head"
        else:
            if m: lid = int(m.group(1)) + 1
            elif "embeddings" in n and "rel_embeddings" not in n: lid = 0
            else: lid = n_layers                        # shared modules (e.g. DeBERTa rel_embeddings) → top-layer LR
            lr, tag = top * cfg["llrd"] ** (n_layers - lid), f"L{lid}"
        wd = 0.0 if (n.endswith(".bias") or "LayerNorm" in n or "layer_norm" in n) else cfg["weight_decay"]
        groups.setdefault((tag, lr, wd), []).append(p)
    pg = [dict(params=v, lr=k[1], weight_decay=k[2]) for k, v in groups.items()]
    return torch.optim.AdamW(pg, betas=(0.9, 0.999), eps=1e-6), sorted({round(g["lr"], 9) for g in pg})

class EMA:
    '''Exponential moving average of trainable weights (kept in fp32 on the training device).'''
    def __init__(self, model, decay):
        self.decay, self.step, self.backup = decay, 0, {}
        self.shadow = {n: p.detach().clone().float() for n, p in model.named_parameters() if p.requires_grad}
    @torch.no_grad()
    def update(self, model):
        self.step += 1; d = min(self.decay, (1 + self.step) / (10 + self.step))
        for n, p in model.named_parameters():
            if n in self.shadow: self.shadow[n].mul_(d).add_(p.detach().float(), alpha=1 - d)
    @torch.no_grad()
    def apply(self, model):
        self.backup = {}
        for n, p in model.named_parameters():
            if n in self.shadow: self.backup[n] = p.detach().clone(); p.copy_(self.shadow[n].to(p.dtype))
    @torch.no_grad()
    def restore(self, model):
        for n, p in model.named_parameters():
            if n in self.backup: p.copy_(self.backup[n])
        self.backup = {}

class FGM:
    '''Fast Gradient Method adversarial perturbation on the word-embedding matrix (optional).'''
    def __init__(self, model, eps=1.0, emb_name="word_embeddings"): self.model, self.eps, self.emb, self.bk = model, eps, emb_name, {}
    def attack(self):
        for n, p in self.model.named_parameters():
            if p.requires_grad and self.emb in n and p.grad is not None:
                norm = torch.norm(p.grad)
                if torch.isfinite(norm) and norm != 0: self.bk[n] = p.data.clone(); p.data.add_(self.eps * p.grad / norm)
    def restore(self):
        for n, p in self.model.named_parameters():
            if n in self.bk: p.data = self.bk[n]
        self.bk = {}

def make_scaler(enabled):
    try: return torch.amp.GradScaler("cuda", enabled=enabled)
    except Exception: return torch.cuda.amp.GradScaler(enabled=enabled)

In [8]:
def train_fold(cfg, key, fold, n_folds, tr_idx, ho_idx, ids_train, ids_test, lengths, sw_all, pad_id, seed, lr_scale, log_params):
    '''Train ONE fold from the pretrained checkpoint; return holdout + test probabilities and statistics.'''
    use_amp = bool(cfg["fp16"]) and DEVICE == "cuda"; set_seed(seed)
    model = load_model(cfg).to(DEVICE)
    pc = log_param_table(key, model, cfg) if log_params else count_params(model)
    opt, lrs = build_optimizer(model, cfg, lr_scale)
    if log_params: log(f"LLRD: {len(lrs)} distinct learning rates from {lrs[0]:.2e} (embeddings) to {lrs[-1]:.2e} (head) | top-layer lr={cfg['lr']*lr_scale:.2e}")
    scaler, ema = make_scaler(use_amp), (EMA(model, cfg["ema_decay"]) if cfg["ema_decay"] else None)
    fgm = FGM(model) if cfg["fgm"] else None
    cw = torch.tensor(compute_class_weight("balanced", classes=np.arange(N_CLASSES), y=y_all[tr_idx]), dtype=torch.float32, device=DEVICE)
    bs, accum, E = cfg["batch_size"], cfg["grad_accum"], cfg["epochs"]
    n_batches = len(make_train_batches(lengths, tr_idx, bs, np.random.default_rng(0)))
    steps_per_epoch = math.ceil(n_batches / accum); total = steps_per_epoch * E; warm = max(1, int(cfg["warmup_frac"] * total))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min((s + 1) / warm, max(0.0, (total - s) / max(1, total - warm))))
    y_t = torch.from_numpy(y_all).to(DEVICE)
    sw_np = sw_all.astype(np.float32)
    sub = np.random.default_rng(seed).choice(ho_idx, min(EPOCH_EVAL_N, len(ho_idx)), replace=False)
    log(f"fold {fold+1}/{n_folds}: train rows={len(tr_idx):,} | holdout rows={len(ho_idx):,} | micro-batch={bs} x accum={accum} (effective {bs*accum}) | "
        f"optimizer steps={total:,} (warmup {warm}) | fp16={use_amp} | EMA={cfg['ema_decay']} | FGM={cfg['fgm']} | seed={seed}")
    t_fold, params_list, bad = time.time(), [p for g in opt.param_groups for p in g["params"]], 0
    for ep in range(E):
        model.train(); t_ep = time.time(); rng = np.random.default_rng(seed * 1000 + ep)
        batches = make_train_batches(lengths, tr_idx, bs, rng); run_loss, seen, step = 0.0, 0, 0; opt.zero_grad(set_to_none=True)
        log_every = max(20, steps_per_epoch // 6)
        for bi, b in enumerate(batches):
            inp, att = collate(ids_train, b, pad_id, DEVICE)
            yb = y_t[torch.from_numpy(b).to(DEVICE)]; swb = torch.from_numpy(sw_np[b]).to(DEVICE)
            with torch.autocast(device_type=DEVICE, dtype=torch.float16, enabled=use_amp):
                logits = model(input_ids=inp, attention_mask=att).logits
            loss = weighted_ce(logits.float(), yb, cw, swb, cfg["label_smoothing"])
            if not torch.isfinite(loss):
                bad += 1
                if bad >= 25: raise FloatingPointError("loss is NaN/Inf for 25 micro-batches — training diverged")
                opt.zero_grad(set_to_none=True); continue
            scaler.scale(loss / accum).backward()
            if fgm is not None:
                fgm.attack()
                with torch.autocast(device_type=DEVICE, dtype=torch.float16, enabled=use_amp):
                    la = model(input_ids=inp, attention_mask=att).logits
                scaler.scale(weighted_ce(la.float(), yb, cw, swb, cfg["label_smoothing"]) / accum).backward(); fgm.restore()
            run_loss += float(loss) * len(b); seen += len(b)
            if (bi + 1) % accum == 0 or bi == len(batches) - 1:
                scaler.unscale_(opt); gn = torch.nn.utils.clip_grad_norm_(params_list, 1.0)
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step(); step += 1
                if ema is not None: ema.update(model)
                if step % log_every == 0 or step == steps_per_epoch:
                    el = time.time() - t_ep
                    log(f"   ep {ep+1}/{E} step {step}/{steps_per_epoch} | loss {run_loss/max(seen,1):.4f} | grad-norm {float(gn):.2f} | "
                        f"lr(top) {sched.get_last_lr()[-1]:.2e} | {seen/el:.0f} samples/s | GPU {gpu_mem()}")
        lg = predict_logits(model, ids_train, sub, pad_id, cfg["eval_bs"], use_amp)
        f1_sub = fast_macro_f1(y_all[sub], lg.argmax(1))
        log(f"   ── epoch {ep+1}/{E} done in {(time.time()-t_ep)/60:.1f} min | train loss {run_loss/max(seen,1):.4f} | "
            f"holdout-subsample ({len(sub)} rows, raw weights) Macro-F1 = {f1_sub:.4f}")
        if ep == 0 and f1_sub < COLLAPSE_F1: raise CollapseError(f"epoch-1 Macro-F1 {f1_sub:.3f} < {COLLAPSE_F1} — run collapsed")
    if ema is not None:
        f_raw = fast_macro_f1(y_all[sub], predict_logits(model, ids_train, sub, pad_id, cfg["eval_bs"], use_amp).argmax(1))
        ema.apply(model)
        f_ema = fast_macro_f1(y_all[sub], predict_logits(model, ids_train, sub, pad_id, cfg["eval_bs"], use_amp).argmax(1))
        log(f"   EMA check on holdout subsample: raw-weights F1={f_raw:.4f} | EMA-weights F1={f_ema:.4f} (EMA weights are used)")
    p_ho = softmax_np(predict_logits(model, ids_train, ho_idx, pad_id, cfg["eval_bs"], use_amp))
    p_te = softmax_np(predict_logits(model, ids_test, np.arange(len(ids_test)), pad_id, cfg["eval_bs"], use_amp))
    stats = dict(seconds=time.time() - t_fold, seed=seed, lr_scale=lr_scale, steps=total, params=pc)
    del model, opt, ema, scaler, fgm; cleanup()
    return p_ho, p_te, stats

def train_fold_safe(cfg, key, fold, n_folds, tr_idx, ho_idx, ids_train, ids_test, lengths, sw_all, pad_id, seed, log_params):
    '''Automatic recovery: collapse/NaN → restart with new seed + lower LR; CUDA OOM → halve micro-batch, double accumulation.'''
    bs_div, lr_scale, seed_off = 1, 1.0, 0
    for attempt in range(3):
        c = dict(cfg); c["batch_size"] = max(1, cfg["batch_size"] // bs_div); c["grad_accum"] = cfg["grad_accum"] * bs_div
        try:
            return train_fold(c, key, fold, n_folds, tr_idx, ho_idx, ids_train, ids_test, lengths, sw_all, pad_id, seed + seed_off, lr_scale, log_params and attempt == 0)
        except (CollapseError, FloatingPointError) as e:
            log(f"fold {fold+1}: {e} → restarting with new seed and lr x0.7 (attempt {attempt+2}/3)", "WARN")
            lr_scale *= 0.7; seed_off += 1000; cleanup()
        except RuntimeError as e:
            if "out of memory" in str(e).lower() and bs_div < 8:
                bs_div *= 2; log(f"fold {fold+1}: CUDA OOM → micro-batch {max(1,cfg['batch_size']//bs_div)} x accum {cfg['grad_accum']*bs_div}", "WARN"); cleanup()
            else: raise
    raise RuntimeError(f"fold {fold+1} failed after 3 attempts")

In [9]:
_fold_seconds = {}   # key → list of measured fold durations this session

def get_noise_weights():
    '''Sample weights from OUT-OF-FOLD predictions of the models trained so far.
       A training row is "confidently contradicted" if the mean OOF probability of its given label is < 0.10 while the predicted
       class has > 0.70. Those rows (max NOISE_MAX_FRACTION of the data) get weight NOISE_WEIGHT; nothing is dropped or relabeled.
       (Second-order note: OOF predictions of row i came from models that saw other rows' labels, the standard caveat of any
       OOF-derived feature; the effect on the reported OOF score is negligible.)'''
    avail = []
    for k in ("xlmr_large", "muril", "mdeberta"):
        st = WORK / f"state_{k}.json"
        if st.exists():
            oof, _, comp = load_model_outputs(k, json.loads(st.read_text())["n_folds"])
            if comp: avail.append(oof)
    sw = np.ones(N_TRAIN, dtype=np.float32)
    if not avail: log("noise weights: no complete transformer OOF yet → all weights = 1.0"); return sw
    P = np.mean(avail, 0); p_true = P[np.arange(N_TRAIN), y_all]; p_max = P.max(1)
    cand = np.where((p_true < 0.10) & (P.argmax(1) != y_all) & (p_max > 0.70))[0]
    cand = cand[np.argsort(p_true[cand])][: int(NOISE_MAX_FRACTION * N_TRAIN)]
    sw[cand] = NOISE_WEIGHT
    log(f"noise weights from {len(avail)} complete OOF model(s): {len(cand)} rows ({len(cand)/N_TRAIN*100:.1f}%) down-weighted to {NOISE_WEIGHT}")
    log("   flagged rows by given label: " + str({LABEL_NAMES[c]: int(n) for c, n in zip(*np.unique(y_all[cand], return_counts=True))}))
    for i in cand[:6]: log(f"   e.g. label={LABEL_NAMES[y_all[i]]!r} model says {LABEL_NAMES[int(P[i].argmax())]!r} ({p_max[i]:.2f}): {train_df['text'].iloc[i][:90]!r}")
    return sw

def stability_probe(cfg, key, tok, ids_train, lengths, pad_id):
    '''Short fp16 dry-run: aborts a backbone that produces NaN losses on this GPU (mDeBERTa can under fp16).'''
    log(f"stability probe for {cfg['pretty']} (30 fp16 steps)…")
    try:
        set_seed(SEED); model = load_model(cfg).to(DEVICE); model.train(); use_amp = cfg["fp16"] and DEVICE == "cuda"
        opt = torch.optim.AdamW(model.parameters(), lr=1e-5); scaler = make_scaler(use_amp); rng = np.random.default_rng(0)
        idx = rng.choice(N_TRAIN, 30 * 16, replace=False); ok = True
        for s in range(30):
            b = idx[s*16:(s+1)*16]; inp, att = collate(ids_train, b, pad_id, DEVICE)
            with torch.autocast(device_type=DEVICE, dtype=torch.float16, enabled=use_amp):
                lg = model(input_ids=inp, attention_mask=att).logits
            loss = F.cross_entropy(lg.float(), torch.from_numpy(y_all[b]).to(DEVICE))
            if not torch.isfinite(loss): ok = False; log(f"probe: non-finite loss at step {s}", "WARN"); break
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
        del model, opt, scaler; cleanup(); return ok
    except Exception:
        log("probe crashed:\n" + traceback.format_exc(), "WARN"); cleanup(); return False

def decide_n_folds(key, cfg, state_path):
    if state_path.exists(): return json.loads(state_path.read_text())["n_folds"]
    k0 = cfg["n_folds"]; measured = _fold_seconds.get(key)
    for k in sorted({k0, 4, 3, 2}, reverse=True):
        if k > k0 or k not in FOLD_SCHEMES: continue
        est = cfg["est_min_per_fold"] * 60 * ((1 - 1 / k) / (1 - 1 / k0)) * k
        log(f"   time check: {k} folds ≈ {est/3600:.2f}h (estimate) vs {time_left_s()/3600:.2f}h available")
        if est < time_left_s():
            state_path.write_text(json.dumps({"n_folds": k})); return k
    return None

def run_transformer_stage(key):
    cfg = MODELS[key]; banner(f"STAGE — {cfg['pretty']} ({cfg['hf_name']})")
    if not RUN[key]: log("stage disabled → skipped"); return
    log("hyper-parameters: " + json.dumps({k: v for k, v in cfg.items() if k not in ("hf_name", "pretty")}))
    try:
        state = WORK / f"state_{key}.json"; K = decide_n_folds(key, cfg, state)
        if K is None: log("not enough time left for this stage → skipped", "WARN"); return
        fcol = f"fold{K}"; log(f"using {K}-fold CV (column {fcol})")
        tok = load_tokenizer(cfg); pad_id = tok.pad_token_id if tok.pad_token_id is not None else 0
        ids_train = tokenize_texts(tok, train_df["text"].tolist(), cfg["max_len"]); ids_test = tokenize_texts(tok, test_df["text"].tolist(), cfg["max_len"])
        lengths = np.array([len(x) for x in ids_train])
        log(f"tokenized: mean length {lengths.mean():.1f} tokens | p99 {np.percentile(lengths, 99):.0f} | truncated at {cfg['max_len']}: {(lengths >= cfg['max_len']).mean()*100:.2f}% of rows | vocab={len(tok):,}")
        if cfg["probe"] and not (WORK / f"probe_{key}.ok").exists():
            if not stability_probe(cfg, key, tok, ids_train, lengths, pad_id):
                log(f"{cfg['pretty']} is numerically unstable in fp16 on this GPU → stage skipped (set fp16=False to force fp32, ~2x slower)", "WARN"); return
            (WORK / f"probe_{key}.ok").write_text("ok"); log("probe passed")
        sw = get_noise_weights() if (cfg["noise_aware"] and USE_NOISE_DOWNWEIGHT) else np.ones(N_TRAIN, dtype=np.float32)
        first = True
        for f in range(K):
            if fold_done(key, f): log(f"fold {f+1}/{K}: already done (loaded from disk) → skipped"); continue
            secs = _fold_seconds.get(key, []); need = (np.mean(secs) if secs else cfg["est_min_per_fold"] * 60) * 1.05
            if time_left_s() < need: log(f"fold {f+1}/{K}: only {time_left_s()/60:.0f} min left (< {need/60:.0f} needed) → stopping stage; it will be excluded from the stack", "WARN"); break
            tr = np.where(train_df[fcol] != f)[0]; ho = np.where(train_df[fcol] == f)[0]
            p_ho, p_te, st = train_fold_safe(cfg, key, f, K, tr, ho, ids_train, ids_test, lengths, sw, pad_id, SEED + f, first); first = False
            _fold_seconds.setdefault(key, []).append(st["seconds"])
            f1 = fast_macro_f1(y_all[ho], p_ho.argmax(1)); per = f1_score(y_all[ho], p_ho.argmax(1), average=None, labels=CLASS_IDS)
            done_n = sum(fold_done(key, g) for g in range(K)) + 1
            log(f"FOLD {f+1}/{K} RESULT: holdout Macro-F1={f1:.4f} (" + ", ".join(f"{LABEL_NAMES[c][:8]} {per[c]:.3f}" for c in CLASS_IDS) + f") | {st['seconds']/60:.1f} min | "
                f"stage ETA ≈ {(K-done_n)*np.mean(_fold_seconds[key])/60:.0f} min | time left {time_left_s()/3600:.2f}h")
            po, pi, pt, pm = fold_files(key, f)
            np.save(po, p_ho.astype(np.float32)); np.save(pi, ho); np.save(pt, p_te.astype(np.float32))
            pm.write_text(json.dumps(dict(f1=f1, params=st["params"], minutes=st["seconds"]/60, seed=st["seed"], lr_scale=st["lr_scale"])))
        report_model(key, K, cfg["pretty"])
    except Exception:
        log(f"{cfg['pretty']} stage FAILED (other stages continue):\n" + traceback.format_exc(), "ERROR"); cleanup()

## 7. Stage 2 — XLM-RoBERTa-large (≈560 M parameters, strongest single model)

In [10]:
run_transformer_stage('xlmr_large')

[04:20:25 | +0h03m] INFO  ========================================================================================
[04:20:25 | +0h03m] INFO  STAGE — XLM-RoBERTa-large (xlm-roberta-large)
[04:20:25 | +0h03m] INFO  ========================================================================================
[04:20:25 | +0h03m] INFO  hyper-parameters: {"n_folds": 4, "epochs": 4, "batch_size": 16, "grad_accum": 2, "eval_bs": 64, "max_len": 128, "lr": 2e-05, "llrd": 0.95, "head_lr_mult": 2.0, "weight_decay": 0.01, "warmup_frac": 0.08, "label_smoothing": 0.05, "ema_decay": null, "fgm": false, "fp16": true, "noise_aware": false, "probe": false, "est_min_per_fold": 38}
[04:20:25 | +0h03m] INFO     time check: 4 folds ≈ 2.53h (estimate) vs 8.27h available
[04:20:25 | +0h03m] INFO  using 4-fold CV (column fold4)


config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

...

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-large
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[05:27:33 | +1h11m] INFO  fold 3/4: train rows=35,803 | holdout rows=11,934 | micro-batch=16 x accum=2 (effective 32) | optimizer steps=4,476 (warmup 358) | fp16=True | EMA=None | FGM=False | seed=44
[05:28:53 | +1h12m] INFO     ep 1/4 step 186/1119 | loss 1.1317 | grad-norm 14.00 | lr(top) 1.04e-05 | 75 samples/s | GPU 6.3/14.6 GB
[05:30:13 | +1h13m] INFO     ep 1/4 step 372/1119 | loss 1.1067 | grad-norm 13.72 | lr(top) 1.99e-05 | 74 samples/s | GPU 6.3/14.6 GB
[05:31:33 | +1h15m] INFO     ep 1/4 step 558/1119 | loss 1.0793 | grad-norm 25.80 | lr(top) 1.90e-05 | 74 samples/s | GPU 6.3/14.6 GB
[05:32:54 | +1h16m] INFO     ep 1/4 step 744/1119 | loss 1.0591 | grad-norm 11.84 | lr(top) 1.81e-05 | 74 samples/s | GPU 6.3/14.6 GB
[05:34:15 | +1h17m] INFO     ep 1/4 step 930/1119 | loss 1.0443 | grad-norm 18.78 | lr(top) 1.72e-05 | 74 samples/s | GPU 6.3/14.6 GB
[05:35:36 | +1h19m] INFO     ep 1/4 step 1116/1119 | loss 1.0324 | grad-norm 18.69 | lr(top) 1.63e-05 | 74 samples/s | GPU 6.3/14.

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-large
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[06:00:28 | +1h44m] INFO  fold 4/4: train rows=35,803 | holdout rows=11,934 | micro-batch=16 x accum=2 (effective 32) | optimizer steps=4,476 (warmup 358) | fp16=True | EMA=None | FGM=False | seed=45
[06:01:49 | +1h45m] INFO     ep 1/4 step 186/1119 | loss 1.1233 | grad-norm 7.64 | lr(top) 1.04e-05 | 74 samples/s | GPU 6.3/14.6 GB
[06:03:10 | +1h46m] INFO     ep 1/4 step 372/1119 | loss 1.1129 | grad-norm 18.61 | lr(top) 1.99e-05 | 74 samples/s | GPU 6.3/14.6 GB
[06:04:31 | +1h48m] INFO     ep 1/4 step 558/1119 | loss 1.0889 | grad-norm 31.67 | lr(top) 1.90e-05 | 74 samples/s | GPU 6.3/14.6 GB
[06:05:51 | +1h49m] INFO     ep 1/4 step 744/1119 | loss 1.0696 | grad-norm 16.34 | lr(top) 1.81e-05 | 74 samples/s | GPU 6.3/14.6 GB
[06:07:12 | +1h50m] INFO     ep 1/4 step 930/1119 | loss 1.0557 | grad-norm 19.89 | lr(top) 1.72e-05 | 74 samples/s | GPU 6.3/14.6 GB
[06:08:33 | +1h52m] INFO     ep 1/4 step 1116/1119 | loss 1.0446 | grad-norm 8.39 | lr(top) 1.63e-05 | 74 samples/s | GPU 6.3/14.6 

## 8. Stage 3 — MuRIL-base (≈237 M parameters)
Google's model pretrained on Indian languages **including Bengali and transliterated/code-mixed text** plus English — the closest match to this dataset's Bangla / English / Banglish mix.
From this stage on, rows that the OOF XLM-R confidently contradicts are down-weighted (noise-aware training).

In [11]:
run_transformer_stage('muril')

[06:33:30 | +2h17m] INFO  ========================================================================================
[06:33:30 | +2h17m] INFO  STAGE — MuRIL-base-cased (google/muril-base-cased)
[06:33:30 | +2h17m] INFO  ========================================================================================
[06:33:30 | +2h17m] INFO  hyper-parameters: {"n_folds": 5, "epochs": 3, "batch_size": 32, "grad_accum": 1, "eval_bs": 128, "max_len": 128, "lr": 3e-05, "llrd": 0.9, "head_lr_mult": 2.0, "weight_decay": 0.01, "warmup_frac": 0.08, "label_smoothing": 0.05, "ema_decay": 0.999, "fgm": false, "fp16": true, "noise_aware": true, "probe": false, "est_min_per_fold": 13}
[06:33:30 | +2h17m] INFO     time check: 5 folds ≈ 1.08h (estimate) vs 6.05h available
[06:33:30 | +2h17m] INFO  using 5-fold CV (column fold5)


config.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/3.16M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/113 [00:00<?, ?B/s]

...

[transformers] BertForSequenceClassification LOAD REPORT from: google/muril-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params w

[07:04:18 | +2h47m] INFO  fold 4/5: train rows=38,190 | holdout rows=9,547 | micro-batch=32 x accum=1 (effective 32) | optimizer steps=3,582 (warmup 286) | fp16=True | EMA=0.999 | FGM=False | seed=45
[07:04:51 | +2h48m] INFO     ep 1/3 step 199/1194 | loss 1.1217 | grad-norm 0.36 | lr(top) 4.20e-05 | 193 samples/s | GPU 3.6/14.6 GB
[07:05:24 | +2h48m] INFO     ep 1/3 step 398/1194 | loss 1.0940 | grad-norm 4.56 | lr(top) 5.80e-05 | 192 samples/s | GPU 3.6/14.6 GB
[07:05:57 | +2h49m] INFO     ep 1/3 step 597/1194 | loss 1.0678 | grad-norm 2.39 | lr(top) 5.43e-05 | 192 samples/s | GPU 3.6/14.6 GB
[07:06:30 | +2h50m] INFO     ep 1/3 step 796/1194 | loss 1.0454 | grad-norm 5.66 | lr(top) 5.07e-05 | 192 samples/s | GPU 3.6/14.6 GB
[07:07:03 | +2h50m] INFO     ep 1/3 step 995/1194 | loss 1.0306 | grad-norm 3.24 | lr(top) 4.71e-05 | 193 samples/s | GPU 3.6/14.6 GB
[07:07:36 | +2h51m] INFO     ep 1/3 step 1194/1194 | loss 1.0166 | grad-norm 6.04 | lr(top) 4.35e-05 | 192 samples/s | GPU 3.6/14.

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google/muril-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params w

[07:14:27 | +2h57m] INFO  fold 5/5: train rows=38,190 | holdout rows=9,547 | micro-batch=32 x accum=1 (effective 32) | optimizer steps=3,582 (warmup 286) | fp16=True | EMA=0.999 | FGM=False | seed=46
[07:15:00 | +2h58m] INFO     ep 1/3 step 199/1194 | loss 1.1208 | grad-norm 0.64 | lr(top) 4.20e-05 | 190 samples/s | GPU 3.6/14.6 GB
[07:15:37 | +2h59m] INFO     ep 1/3 step 398/1194 | loss 1.0884 | grad-norm 8.26 | lr(top) 5.80e-05 | 181 samples/s | GPU 3.6/14.6 GB
[07:16:21 | +2h59m] INFO     ep 1/3 step 597/1194 | loss 1.0634 | grad-norm 10.38 | lr(top) 5.43e-05 | 167 samples/s | GPU 3.6/14.6 GB
[07:16:57 | +3h00m] INFO     ep 1/3 step 796/1194 | loss 1.0451 | grad-norm inf | lr(top) 5.07e-05 | 169 samples/s | GPU 3.6/14.6 GB
[07:17:30 | +3h01m] INFO     ep 1/3 step 995/1194 | loss 1.0274 | grad-norm 6.59 | lr(top) 4.71e-05 | 174 samples/s | GPU 3.6/14.6 GB
[07:18:03 | +3h01m] INFO     ep 1/3 step 1194/1194 | loss 1.0122 | grad-norm 6.44 | lr(top) 4.35e-05 | 176 samples/s | GPU 3.6/14.

## 9. Stage 4 — mDeBERTa-v3-base (≈280 M parameters)
Multilingual DeBERTa-v3 (disentangled attention, replaced-token-detection pretraining) — often stronger than XLM-R at base size and architecturally very different.
Guarded by a 30-step fp16 stability probe; skipped automatically if it produces NaNs on this GPU.

In [12]:
run_transformer_stage('mdeberta')

[07:24:52 | +3h08m] INFO  ========================================================================================
[07:24:52 | +3h08m] INFO  STAGE — mDeBERTa-v3-base (microsoft/mdeberta-v3-base)
[07:24:52 | +3h08m] INFO  ========================================================================================
[07:24:52 | +3h08m] INFO  hyper-parameters: {"n_folds": 5, "epochs": 3, "batch_size": 32, "grad_accum": 1, "eval_bs": 128, "max_len": 128, "lr": 3e-05, "llrd": 0.9, "head_lr_mult": 2.0, "weight_decay": 0.01, "warmup_frac": 0.08, "label_smoothing": 0.05, "ema_decay": 0.999, "fgm": false, "fp16": true, "noise_aware": true, "probe": true, "est_min_per_fold": 18}
[07:24:52 | +3h08m] INFO     time check: 5 folds ≈ 1.50h (estimate) vs 5.19h available
[07:24:52 | +3h08m] INFO  using 5-fold CV (column fold5)


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 4.31MB            

spm.model: downloading bytes:           |  0.00B            

...

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/mdeberta-v3-base
Key                                        | Status     | 
-------------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.weight    | UNEXPECTED | 
mask_predictions.LayerNorm.weight          | UNEXPECTED | 
lm_predictions.lm_head.dense.weight        | UNEXPECTED | 
mask_predictions.dense.bias                | UNEXPECTED | 
lm_predictions.lm_head.bias                | UNEXPECTED | 
mask_predictions.classifier.bias           | UNEXPECTED | 
mask_predictions.classifier.weight         | UNEXPECTED | 
mask_predictions.LayerNorm.bias            | UNEXPECTED | 
mask_predictions.dense.weight              | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED | 
lm_predictions.lm_head.dense.bias          | UNEXPECTED | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias     

[08:10:04 | +3h53m] INFO  fold 4/5: train rows=38,190 | holdout rows=9,547 | micro-batch=32 x accum=1 (effective 32) | optimizer steps=3,582 (warmup 286) | fp16=True | EMA=0.999 | FGM=False | seed=45
[08:10:52 | +3h54m] INFO     ep 1/3 step 199/1194 | loss 1.1086 | grad-norm 4.70 | lr(top) 4.20e-05 | 134 samples/s | GPU 4.2/14.6 GB
[08:11:40 | +3h55m] INFO     ep 1/3 step 398/1194 | loss 1.0613 | grad-norm 6.67 | lr(top) 5.80e-05 | 133 samples/s | GPU 4.2/14.6 GB
[08:12:28 | +3h56m] INFO     ep 1/3 step 597/1194 | loss 1.0343 | grad-norm 7.48 | lr(top) 5.43e-05 | 133 samples/s | GPU 4.2/14.6 GB
[08:13:16 | +3h56m] INFO     ep 1/3 step 796/1194 | loss 1.0158 | grad-norm 6.68 | lr(top) 5.07e-05 | 133 samples/s | GPU 4.2/14.6 GB
[08:14:03 | +3h57m] INFO     ep 1/3 step 995/1194 | loss 1.0043 | grad-norm 4.41 | lr(top) 4.71e-05 | 134 samples/s | GPU 4.2/14.6 GB
[08:14:52 | +3h58m] INFO     ep 1/3 step 1194/1194 | loss 0.9950 | grad-norm 3.91 | lr(top) 4.35e-05 | 133 samples/s | GPU 4.2/14.

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/mdeberta-v3-base
Key                                        | Status     | 
-------------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.weight    | UNEXPECTED | 
mask_predictions.LayerNorm.weight          | UNEXPECTED | 
lm_predictions.lm_head.dense.weight        | UNEXPECTED | 
mask_predictions.dense.bias                | UNEXPECTED | 
lm_predictions.lm_head.bias                | UNEXPECTED | 
mask_predictions.classifier.bias           | UNEXPECTED | 
mask_predictions.classifier.weight         | UNEXPECTED | 
mask_predictions.LayerNorm.bias            | UNEXPECTED | 
mask_predictions.dense.weight              | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED | 
lm_predictions.lm_head.dense.bias          | UNEXPECTED | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias     

[08:24:57 | +4h08m] INFO  fold 5/5: train rows=38,190 | holdout rows=9,547 | micro-batch=32 x accum=1 (effective 32) | optimizer steps=3,582 (warmup 286) | fp16=True | EMA=0.999 | FGM=False | seed=46
[08:25:46 | +4h09m] INFO     ep 1/3 step 199/1194 | loss 1.1119 | grad-norm 4.46 | lr(top) 4.20e-05 | 132 samples/s | GPU 4.2/14.6 GB
[08:26:33 | +4h10m] INFO     ep 1/3 step 398/1194 | loss 1.0709 | grad-norm 9.82 | lr(top) 5.80e-05 | 133 samples/s | GPU 4.2/14.6 GB
[08:27:20 | +4h10m] INFO     ep 1/3 step 597/1194 | loss 1.0466 | grad-norm 7.44 | lr(top) 5.43e-05 | 134 samples/s | GPU 4.2/14.6 GB
[08:28:07 | +4h11m] INFO     ep 1/3 step 796/1194 | loss 1.0253 | grad-norm 8.27 | lr(top) 5.07e-05 | 134 samples/s | GPU 4.2/14.6 GB
[08:28:55 | +4h12m] INFO     ep 1/3 step 995/1194 | loss 1.0109 | grad-norm 6.15 | lr(top) 4.71e-05 | 134 samples/s | GPU 4.2/14.6 GB
[08:29:42 | +4h13m] INFO     ep 1/3 step 1194/1194 | loss 0.9982 | grad-norm 4.42 | lr(top) 4.35e-05 | 134 samples/s | GPU 4.2/14.

## 10. Stacking, thresholds and final submissions
1. Collect every **complete** model's OOF probabilities (incomplete stages are excluded automatically).
2. Baselines: simple mean and weight-searched blend of the models.
3. **Stackers** (multinomial logistic regression, gradient-boosted trees, and their average) on `log(prob)` of every model + text meta-features (script, length, emoji, casing, punctuation, URL/@ flags), validated with 5-fold CV over the OOF rows.
4. **Per-class thresholds** maximising Macro-F1, scored with *nested* CV (thresholds tuned on 4/5 of the rows, evaluated on the other 1/5) so the headline number is not inflated.
5. Note: test-set probabilities are the *average of the K fold models* (slightly smoother than a single-model OOF prediction) — the usual, accepted stacking convention.
6. Two submissions are written — **A** (best stack + thresholds) and **B** (weight-searched blend + thresholds, more conservative) — the two you select on Kaggle.

In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import itertools

EMOJI_RE = re.compile("[\U0001F300-\U0001FAFF\u2600-\u27BF\U0001F1E6-\U0001F1FF]")
def meta_features(df):
    t = df["text"]; o = pd.DataFrame(index=df.index)
    o["log_len"] = np.log1p(df["char_len"]); o["log_words"] = np.log1p(df["word_count"])
    for s in ("bangla", "latin", "mixed", "other"): o["script_" + s] = (df["script"] == s).astype(float)
    o["n_excl"] = t.str.count("!").clip(upper=10); o["n_q"] = t.str.count(r"\?").clip(upper=10)
    o["upper_ratio"] = t.apply(lambda s: sum(c.isupper() for c in s) / max(1, sum(c.isalpha() for c in s)))
    o["n_emoji"] = t.apply(lambda s: len(EMOJI_RE.findall(s))).clip(upper=10)
    o["has_url"] = t.str.contains(r"\[URL\]", regex=True).astype(float); o["has_user"] = t.str.contains(r"\[USER\]", regex=True).astype(float)
    return o.values.astype(np.float32)

def run_final():
    banner("FINAL — stacking, thresholds, submissions")
    models = {}   # key → (oof, test_mean)
    for key, K, pretty in [("svm", SVM_CFG["n_folds"], "TF-IDF+SVM")] + \
            [(k, json.loads((WORK / f"state_{k}.json").read_text())["n_folds"] if (WORK / f"state_{k}.json").exists() else 0, MODELS[k]["pretty"]) for k in MODELS]:
        if K == 0: log(f"{pretty}: not run → excluded", "WARN"); continue
        oof, te, comp = load_model_outputs(key, K)
        if comp: models[key] = (oof, te)
        else: log(f"{pretty}: incomplete → excluded", "WARN")
    if not models: raise RuntimeError("no complete model outputs found — run the stages first")
    names = list(models)
    rows = [{"model": k, "OOF Macro-F1": fast_macro_f1(y_all, models[k][0].argmax(1))} for k in names]
    tf_names = [k for k in names if k != "svm"]
    log("models in the stack: " + ", ".join(names))

    # ── blends ──
    def blend(ws, which):  # which: 0 → oof, 1 → test
        return sum(w * models[k][which] for w, k in zip(ws, names))
    mean_ws = [1 / len(names)] * len(names)
    best_ws, best_f = mean_ws, fast_macro_f1(y_all, blend(mean_ws, 0).argmax(1))
    grid = np.round(np.arange(0, 1.0001, 0.1), 2)
    for ws in itertools.product(grid, repeat=len(names)):
        if abs(sum(ws) - 1) > 1e-6: continue
        f = fast_macro_f1(y_all, blend(ws, 0).argmax(1))
        if f > best_f: best_f, best_ws = f, list(ws)
    log("blend weights (grid search on OOF): " + str({k: round(float(w), 2) for k, w in zip(names, best_ws)}))
    B_oof, B_te = blend(best_ws, 0), blend(best_ws, 1)
    rows += [{"model": "mean blend", "OOF Macro-F1": fast_macro_f1(y_all, blend(mean_ws, 0).argmax(1))},
             {"model": "weighted blend", "OOF Macro-F1": best_f}]

    # ── stackers ──
    Xtr_meta, Xte_meta = meta_features(train_df), meta_features(test_df)
    lg = lambda P: np.log(np.clip(P, 1e-6, 1))
    Xtr = np.hstack([lg(models[k][0]) for k in names] + [Xtr_meta]); Xte = np.hstack([lg(models[k][1]) for k in names] + [Xte_meta])
    log(f"stacker input: {Xtr.shape[1]} features = {len(names)} models x 3 log-probs + {Xtr_meta.shape[1]} meta-features")
    cvs = list(StratifiedKFold(5, shuffle=True, random_state=SEED + 99).split(Xtr, y_all))
    bal = compute_class_weight("balanced", classes=np.arange(3), y=y_all)[y_all]
    def cv_oof(make):
        P = np.zeros((N_TRAIN, 3))
        for tr, te in cvs:
            m = make().fit(Xtr[tr], y_all[tr]); P[te] = m.predict_proba(Xtr[te])
        return P
    cand = {}
    best_lr = None
    for C in (0.03, 0.1, 0.3, 1.0):
        mk = lambda C=C: make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=3000, class_weight="balanced"))
        P = cv_oof(mk); f = fast_macro_f1(y_all, P.argmax(1)); log(f"   LogReg C={C}: CV Macro-F1={f:.4f}")
        if best_lr is None or f > best_lr[0]: best_lr = (f, C, P, mk)
    cand["stack_logreg"] = (best_lr[2], best_lr[3])
    mk_h = lambda: HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=250, l2_regularization=1.0, random_state=SEED)
    P_h = np.zeros((N_TRAIN, 3))
    for tr, te in cvs: m = mk_h().fit(Xtr[tr], y_all[tr], sample_weight=bal[tr]); P_h[te] = m.predict_proba(Xtr[te])
    log(f"   HistGradientBoosting: CV Macro-F1={fast_macro_f1(y_all, P_h.argmax(1)):.4f}")
    cand["stack_hgb"] = (P_h, mk_h); cand["stack_avg"] = ((best_lr[2] + P_h) / 2, None)
    for k, (P, _) in cand.items(): rows.append({"model": k, "OOF Macro-F1": fast_macro_f1(y_all, P.argmax(1))})

    # ── thresholds with nested CV, pick winners ──
    summary = []
    for k, P in [("weighted blend", B_oof)] + [(k, v[0]) for k, v in cand.items()]:
        raw = fast_macro_f1(y_all, P.argmax(1)); nested = nested_threshold_estimate(P, y_all)
        summary.append((k, raw, nested)); log(f"{k:<16} raw OOF F1={raw:.4f} | + thresholds (nested-CV honest estimate)={nested:.4f}")
    best_stack = max([s for s in summary if s[0].startswith("stack")], key=lambda s: s[2])[0]
    log(f"best stacker by nested estimate: {best_stack}")

    # final fits → test probabilities
    def fit_stack(name):
        if name == "stack_logreg": return make_pipeline(StandardScaler(), LogisticRegression(C=best_lr[1], max_iter=3000, class_weight="balanced")).fit(Xtr, y_all).predict_proba(Xte)
        if name == "stack_hgb": return mk_h().fit(Xtr, y_all, sample_weight=bal).predict_proba(Xte)
        a = make_pipeline(StandardScaler(), LogisticRegression(C=best_lr[1], max_iter=3000, class_weight="balanced")).fit(Xtr, y_all).predict_proba(Xte)
        return (a + mk_h().fit(Xtr, y_all, sample_weight=bal).predict_proba(Xte)) / 2
    S_oof, S_te = cand[best_stack][0], fit_stack(best_stack)
    wA = persistent_thresholds(S_oof, y_all, f"A_{best_stack}_{'+'.join(names)}")
    wB = persistent_thresholds(B_oof, y_all, f"B_blend_{'+'.join(names)}")
    predA, predB = apply_weights(S_te, wA), apply_weights(B_te, wB)
    log(f"submission A on OOF with the saved thresholds (in-sample, slightly optimistic): Macro-F1 = {fast_macro_f1(y_all, apply_weights(S_oof, wA)):.4f}")

    ids = test_df["id"].to_numpy()
    for nm, pr in (("submission_A_stack_thresholds.csv", predA), ("submission_B_blend_thresholds.csv", predB)):
        sub = pd.DataFrame({"id": ids, "y_pred": pr.astype(int)}); assert sub["y_pred"].isin(CLASS_IDS).all() and len(sub) == N_TEST
        for d in (WORK / "submissions", Path("/content/submissions")):
            try: d.mkdir(parents=True, exist_ok=True); sub.to_csv(d / nm, index=False)
            except Exception: pass
        dist = sub["y_pred"].value_counts(normalize=True).sort_index()
        log(f"wrote {nm} | predicted mix: " + " | ".join(f"{LABEL_NAMES[c][:8]}={dist.get(c,0):.3f}" for c in CLASS_IDS) + " | train prior: " + " | ".join(f"{p:.3f}" for p in PRIOR))
    try:
        ss = pd.read_csv(find_file("sample_submission.csv")); assert list(ss.columns) == ["id", "y_pred"]
        if not SMOKE_TEST: assert set(ss["id"]) == set(ids), "test ids differ from sample_submission"; log("format check vs sample_submission.csv: OK")
    except FileNotFoundError: log("sample_submission.csv not found — format check skipped", "WARN")
    A_nested = [s for s in summary if s[0] == best_stack][0][2]; B_nested = [s for s in summary if s[0] == "weighted blend"][0][2]
    tab = pd.DataFrame(rows).drop_duplicates("model")
    log("\nRESULTS TABLE (OOF over all labeled rows)\n" + tab.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
    log(f"HEADLINE (nested-CV honest estimates): A = {A_nested:.4f} | B = {B_nested:.4f}")
    return dict(names=names, tab=tab, A=A_nested, B=B_nested, best_stack=best_stack)

FINAL = run_final()

[08:39:45 | +4h23m] INFO  ========================================================================================
[08:39:45 | +4h23m] INFO  FINAL — stacking, thresholds, submissions
[08:39:45 | +4h23m] INFO  ========================================================================================
[08:39:45 | +4h23m] INFO  models in the stack: svm, xlmr_large, muril, mdeberta
[08:39:46 | +4h23m] INFO  blend weights (grid search on OOF): {'svm': 0.5, 'xlmr_large': 0.3, 'muril': 0.1, 'mdeberta': 0.1}
[08:39:46 | +4h23m] INFO  stacker input: 24 features = 4 models x 3 log-probs + 12 meta-features
[08:39:49 | +4h23m] INFO     LogReg C=0.03: CV Macro-F1=0.5758
[08:39:53 | +4h23m] INFO     LogReg C=0.1: CV Macro-F1=0.5755
[08:39:56 | +4h23m] INFO     LogReg C=0.3: CV Macro-F1=0.5754
[08:39:59 | +4h23m] INFO     LogReg C=1.0: CV Macro-F1=0.5755
[08:40:08 | +4h23m] INFO     HistGradientBoosting: CV Macro-F1=0.5751
[08:40:19 | +4h23m] INFO  weighted blend   raw OOF F1=0.5963 | + thresholds (nest

## 11. Methodology table (auto-generated: architecture, parameters, hyper-parameters actually used)

In [14]:
def methodology():
    rows = []
    sj = WORK / "svm_best_C.json"
    rows.append(dict(component="TF-IDF + LinearSVC (Platt-calibrated)", architecture=f"word 1-2g + char_wb 3-5g TF-IDF (≤{SVM_CFG['max_features_each']*2:,} feats), linear SVM",
                     params="≈0.4M weights x3 calibrated copies", folds=SVM_CFG["n_folds"],
                     hyperparams=f"C={json.loads(sj.read_text())['C'] if sj.exists() else 'n/a'}, class_weight=balanced, sigmoid calibration cv=3"))
    for k, c in MODELS.items():
        st = WORK / f"state_{k}.json"; nf = json.loads(st.read_text())["n_folds"] if st.exists() else None
        meta = WORK / "folds" / f"{k}_f0_meta.json"; pc = json.loads(meta.read_text()).get("params") if meta.exists() else None
        rows.append(dict(component=c["pretty"], architecture=f"{c['hf_name']} + classification head",
                         params=(f"{pc['total']/1e6:.0f}M total ({pc['embeddings']/1e6:.0f}M embeddings, {pc['encoder']/1e6:.0f}M layers, {pc['head']/1e6:.1f}M head)" if pc else "n/a"),
                         folds=nf,
                         hyperparams=f"epochs={c['epochs']}, batch={c['batch_size']}x{c['grad_accum']}, max_len={c['max_len']}, lr(top)={c['lr']}, LLRD={c['llrd']}, head_lr x{c['head_lr_mult']}, "
                                     f"wd={c['weight_decay']}, warmup={c['warmup_frac']}, label_smoothing={c['label_smoothing']}, EMA={c['ema_decay']}, fp16={c['fp16']}, noise-aware={c['noise_aware']}"))
    rows.append(dict(component="Stacker", architecture="multinomial LogReg / HistGradientBoosting on log-probs + text meta-features",
                     params="a few hundred coefficients / 250 depth-3 trees", folds="5 (CV over OOF rows)", hyperparams=f"best={FINAL['best_stack'] if FINAL else 'n/a'}, per-class thresholds by grid search (nested-CV validated)"))
    df = pd.DataFrame(rows)
    try: md_text = "# Methodology\n\n" + df.to_markdown(index=False)
    except Exception: md_text = "# Methodology\n\n" + df.to_string(index=False)
    try: (WORK / "METHODOLOGY.md").write_text(md_text)
    except Exception: pass
    log("METHODOLOGY\n" + df.to_string(index=False, max_colwidth=140))
    return df

try: METH = methodology()
except Exception: log("methodology table failed:\n" + traceback.format_exc(), "WARN")
banner(f"DONE in {(time.time()-T0)/3600:.2f}h — submit  submission_A_stack_thresholds.csv  and  submission_B_blend_thresholds.csv  (folder: {WORK/'submissions'})")

[08:41:00 | +4h24m] INFO  METHODOLOGY
                            component                                                                architecture                                               params                folds                                                                                                                                  hyperparams
TF-IDF + LinearSVC (Platt-calibrated)                word 1-2g + char_wb 3-5g TF-IDF (≤120,000 feats), linear SVM                   ≈0.4M weights x3 calibrated copies                    5                                                                                       C=0.1, class_weight=balanced, sigmoid calibration cv=3
                    XLM-RoBERTa-large                                     xlm-roberta-large + classification head 560M total (256M embeddings, 303M layers, 1.1M head)                    4 epochs=4, batch=16x2, max_len=128, lr(top)=2e-05, LLRD=0.95, head_lr x2.0, wd=0.01, warmup=0.08, label_smoothing=0.0

zip download


In [15]:
# ── Download final submissions + key artifacts to your local machine ─────────
import shutil
from google.colab import files

sub_dir = WORK / "submissions"
print("📂 Files in submissions folder:")
for p in sorted(sub_dir.glob("*")):
    print(f"   {p.name} ({p.stat().st_size/1024:.1f} KB)")

# Individual files (the two you select on Kaggle)
for name in ["submission_A_stack_thresholds.csv", "submission_B_blend_thresholds.csv"]:
    p = sub_dir / name
    if p.exists():
        files.download(str(p))
    else:
        print(f"⚠️ {name} not found — did run_final() finish?")

# Backup zip: submissions + thresholds + methodology + run log
extras = ["thresholds.json", "METHODOLOGY.md", "run_log.txt"]
bundle = Path("/content/final_bundle"); bundle.mkdir(exist_ok=True)
for p in sub_dir.glob("*.csv"): shutil.copy(p, bundle / p.name)
for e in extras:
    if (WORK / e).exists(): shutil.copy(WORK / e, bundle / e)
zip_path = shutil.make_archive("/content/final_bundle", "zip", bundle)
files.download(zip_path)

📂 Files in submissions folder:
   submission_A_stack_thresholds.csv (91.3 KB)
   submission_B_blend_thresholds.csv (91.3 KB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [17]:
# ═══════════════ ADD-ON: text-twin override — runs in SECONDS, CPU only ═══════════════
# ~3% of test comments have an identical twin in train (same text after lower-casing / stripping punctuation, URLs, @mentions).
# The models have seen those twins during training, but they don't always reproduce the twin's label. This checks — on the
# out-of-fold predictions — whether copying the twin's label beats the model on twin rows, using leave-one-out on train
# (each train row's twin label comes from the OTHER train rows only). It is applied ONLY if it improves Macro-F1.
# It uses the comment text only (no ids, no external data): a nearest-neighbour lookup on the text.
def run_twin_override(min_gain=0.001):
    import itertools, glob
    from collections import defaultdict
    banner("ADD-ON — text-twin override")
    def key(t):
        t = unicodedata.normalize("NFC", str(t)).lower(); t = re.sub(r"\[url\]|\[user\]", " ", t)
        return WS_RE.sub(" ", re.sub(r"[^\w\s]", " ", t)).strip()
    ktr, kte = train_df["text"].map(key).to_numpy(), test_df["text"].map(key).to_numpy()
    groups = defaultdict(list)
    for i, k in enumerate(ktr):
        if len(k) >= 2: groups[k].append(i)          # ignore empty / 1-char keys (emoji-only, "?", ...)
    cnt_tr = np.zeros((N_TRAIN, 3), int); cnt_te = np.zeros((N_TEST, 3), int)
    for i, k in enumerate(ktr):
        if k in groups and len(groups[k]) > 1: cnt_tr[i] = np.bincount(y_all[[j for j in groups[k] if j != i]], minlength=3)
    for j, k in enumerate(kte):
        if k in groups: cnt_te[j] = np.bincount(y_all[groups[k]], minlength=3)
    n_tr, n_te = cnt_tr.sum(1), cnt_te.sum(1)
    log(f"rows with >=1 twin: train {int((n_tr>0).sum()):,} ({(n_tr>0).mean()*100:.1f}%) | test {int((n_te>0).sum()):,} ({(n_te>0).mean()*100:.1f}%)")

    # base predictions = weighted blend of all complete models' OOF + tuned thresholds
    models = {}
    for kname, K in [("svm", SVM_CFG["n_folds"])] + [(k, json.loads((WORK / f"state_{k}.json").read_text())["n_folds"]) for k in MODELS if (WORK / f"state_{k}.json").exists()]:
        oof, te, comp = load_model_outputs(kname, K)
        if comp: models[kname] = oof
    names = list(models); best_f, best_ws = -1, None
    for ws in itertools.product(np.round(np.arange(0, 1.0001, 0.1), 2), repeat=len(names)):
        if abs(sum(ws) - 1) > 1e-6: continue
        f = fast_macro_f1(y_all, sum(w * models[k] for w, k in zip(ws, names)).argmax(1))
        if f > best_f: best_f, best_ws = f, ws
    B = sum(w * models[k] for w, k in zip(best_ws, names)); wthr, _ = tune_class_weights(B, y_all); base = apply_weights(B, wthr)
    f_base = fast_macro_f1(y_all, base)

    def rule_pred(cnt, n, pred, rule):
        out = pred.copy(); top = cnt.argmax(1); mx = cnt.max(1)
        tie = (cnt == mx[:, None]).sum(1) > 1; unan = (mx == n) & (n > 0)
        m = {"majority": (n > 0) & ~tie, "unanimous": unan, "unanimous_2plus": unan & (n >= 2)}[rule]
        out[m] = top[m]; return out, int(m.sum())
    has = n_tr > 0
    log(f"on the {int(has.sum()):,} train twin rows: model accuracy = {(base[has]==y_all[has]).mean():.3f} | majority-twin-label accuracy = {(cnt_tr[has].argmax(1)==y_all[has]).mean():.3f}")
    best_rule, best_gain = None, min_gain
    for rule in ("majority", "unanimous", "unanimous_2plus"):
        p, m = rule_pred(cnt_tr, n_tr, base, rule); g = fast_macro_f1(y_all, p) - f_base
        log(f"   rule {rule:<16}: overrides {m:,} train rows | OOF Macro-F1 {f_base:.4f} -> {f_base+g:.4f} ({g:+.4f})")
        if g > best_gain: best_rule, best_gain = rule, g
    if best_rule is None: log(f"no rule gains >= {min_gain} → twin override NOT applied (model already handles twins). Nothing written."); return None
    log(f"applying rule '{best_rule}' (OOF gain {best_gain:+.4f}) to every submission file")
    for path in sorted(glob.glob(str(WORK / "submissions" / "submission_*.csv"))):
        if path.endswith(("_twin.csv", "_v1.csv")): continue
        sub = pd.read_csv(path); pos = pd.Series(np.arange(N_TEST), index=test_df["id"].to_numpy()).loc[sub["id"]].to_numpy()
        newp, m = rule_pred(cnt_te[pos], n_te[pos], sub["y_pred"].to_numpy(), best_rule)
        ch = int((newp != sub["y_pred"].to_numpy()).sum()); out = sub.copy(); out["y_pred"] = newp.astype(int)
        for d in (WORK / "submissions", Path("/content/submissions")):
            try: d.mkdir(parents=True, exist_ok=True); out.to_csv(d / (Path(path).stem + "_twin.csv"), index=False)
            except Exception: pass
        log(f"wrote {Path(path).stem}_twin.csv | test rows overridden={m} | labels actually changed={ch}")
    return best_gain

TWIN_GAIN = run_twin_override()

[08:56:25 | +4h39m] INFO  ========================================================================================
[08:56:25 | +4h39m] INFO  ADD-ON — text-twin override
[08:56:25 | +4h39m] INFO  ========================================================================================
[08:56:27 | +4h40m] INFO  rows with >=1 twin: train 1,469 (3.1%) | test 396 (3.3%)
[08:56:34 | +4h40m] INFO  on the 1,469 train twin rows: model accuracy = 0.790 | majority-twin-label accuracy = 0.875
[08:56:34 | +4h40m] INFO     rule majority        : overrides 1,409 train rows | OOF Macro-F1 0.6009 -> 0.6028 (+0.0019)
[08:56:34 | +4h40m] INFO     rule unanimous       : overrides 1,352 train rows | OOF Macro-F1 0.6009 -> 0.6027 (+0.0019)
[08:56:34 | +4h40m] INFO     rule unanimous_2plus : overrides 310 train rows | OOF Macro-F1 0.6009 -> 0.6008 (-0.0000)
[08:56:34 | +4h40m] INFO  applying rule 'majority' (OOF gain +0.0019) to every submission file
[08:56:34 | +4h40m] INFO  wrote submission_A_stack_threshol

In [18]:
# ═══════════════ ADD-ON: per-script thresholds (Bangla vs Latin) on the weighted blend — CPU only, ~1 min ═══════════════
# Bangla and English comments have different class balances and different model reliability, so one global set of
# class thresholds is a compromise. This tunes separate thresholds per script group and keeps them ONLY if the
# nested-CV estimate (thresholds tuned on 4/5 of rows, scored on the other 1/5) beats the global-threshold estimate.
def run_script_thresholds(min_gain=0.002):
    import itertools
    banner("ADD-ON — per-script thresholds on the weighted blend")
    models = {}
    for key, K in [("svm", SVM_CFG["n_folds"])] + [(k, json.loads((WORK / f"state_{k}.json").read_text())["n_folds"])
                                                   for k in MODELS if (WORK / f"state_{k}.json").exists()]:
        oof, te, comp = load_model_outputs(key, K)
        if comp: models[key] = (oof, te)
    names = list(models); log("models in blend: " + ", ".join(names))
    best_f, best_ws = -1, None
    for ws in itertools.product(np.round(np.arange(0, 1.0001, 0.1), 2), repeat=len(names)):
        if abs(sum(ws) - 1) > 1e-6: continue
        f = fast_macro_f1(y_all, sum(w * models[k][0] for w, k in zip(ws, names)).argmax(1))
        if f > best_f: best_f, best_ws = f, ws
    log("blend weights: " + str({k: round(float(w), 2) for k, w in zip(names, best_ws)}) + f" | raw OOF F1 {best_f:.4f}")
    B_oof = sum(w * models[k][0] for w, k in zip(best_ws, names)); B_te = sum(w * models[k][1] for w, k in zip(best_ws, names))
    g_tr = (train_df["script"] == "bangla").to_numpy(); g_te = (test_df["script"] == "bangla").to_numpy()
    log(f"rows: bangla-script={int(g_tr.sum()):,} | latin/mixed/other={int((~g_tr).sum()):,}")
    MIN_ROWS = 300; pg = np.zeros(N_TRAIN, int); pl = np.zeros(N_TRAIN, int)
    for tr, te in StratifiedKFold(5, shuffle=True, random_state=123).split(B_oof, y_all * 2 + g_tr):
        w, _ = tune_class_weights(B_oof[tr], y_all[tr]); pg[te] = apply_weights(B_oof[te], w)
        for g in (True, False):
            a, b = tr[g_tr[tr] == g], te[g_tr[te] == g]
            wg = tune_class_weights(B_oof[a], y_all[a])[0] if len(a) >= MIN_ROWS else w
            if len(b): pl[b] = apply_weights(B_oof[b], wg)
    f_glob, f_grp = fast_macro_f1(y_all, pg), fast_macro_f1(y_all, pl)
    for g, nm in ((True, "bangla"), (False, "latin+")):
        log(f"   {nm:<7} nested F1: global thresholds {fast_macro_f1(y_all[g_tr == g], pg[g_tr == g]):.4f} | per-script thresholds {fast_macro_f1(y_all[g_tr == g], pl[g_tr == g]):.4f}")
    log(f"NESTED ESTIMATE  global thresholds = {f_glob:.4f} | per-script thresholds = {f_grp:.4f} | gain = {f_grp - f_glob:+.4f}")
    if f_grp < f_glob + min_gain:
        log(f"gain < {min_gain} → NOT worth it (would be fitting noise). Keep submissions A/B; no file written."); return None
    pred = np.zeros(N_TEST, int)
    for g in (True, False):
        wg = tune_class_weights(B_oof[g_tr == g], y_all[g_tr == g])[0]
        log(f"final thresholds {'bangla' if g else 'latin+'}: {np.round(wg, 3).tolist()}"); pred[g_te == g] = apply_weights(B_te[g_te == g], wg)
    sub = pd.DataFrame({"id": test_df["id"].to_numpy(), "y_pred": pred.astype(int)}); assert len(sub) == N_TEST and sub["y_pred"].isin(CLASS_IDS).all()
    for d in (WORK / "submissions", Path("/content/submissions")):
        try: d.mkdir(parents=True, exist_ok=True); sub.to_csv(d / "submission_C_blend_script_thresholds.csv", index=False)
        except Exception: pass
    dist = sub["y_pred"].value_counts(normalize=True).sort_index()
    log("wrote submission_C_blend_script_thresholds.csv | mix: " + " | ".join(f"{dist.get(c,0):.3f}" for c in CLASS_IDS))
    return f_grp

SCRIPT_C = run_script_thresholds()

[08:56:40 | +4h40m] INFO  ========================================================================================
[08:56:40 | +4h40m] INFO  ADD-ON — per-script thresholds on the weighted blend
[08:56:40 | +4h40m] INFO  ========================================================================================
[08:56:40 | +4h40m] INFO  models in blend: svm, xlmr_large, muril, mdeberta
[08:56:41 | +4h40m] INFO  blend weights: {'svm': 0.5, 'xlmr_large': 0.3, 'muril': 0.1, 'mdeberta': 0.1} | raw OOF F1 0.5963
[08:56:41 | +4h40m] INFO  rows: bangla-script=19,136 | latin/mixed/other=28,601
[08:57:31 | +4h41m] INFO     bangla  nested F1: global thresholds 0.5105 | per-script thresholds 0.5126
[08:57:31 | +4h41m] INFO     latin+  nested F1: global thresholds 0.6262 | per-script thresholds 0.6266
[08:57:31 | +4h41m] INFO  NESTED ESTIMATE  global thresholds = 0.5999 | per-script thresholds = 0.5960 | gain = -0.0039
[08:57:31 | +4h41m] INFO  gain < 0.002 → NOT worth it (would be fitting noise). Kee